In [1]:
import pandas as pd
from rapidfuzz import process, fuzz
import requests
from bs4 import BeautifulSoup

In [2]:
# Exploración de los dataset
df_global = pd.read_csv("../data/raw/global_football_results.csv")
df_global.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [3]:
# Primer análisis
df_global.info()

<class 'pandas.DataFrame'>
RangeIndex: 47399 entries, 0 to 47398
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        47399 non-null  str  
 1   home_team   47399 non-null  str  
 2   away_team   47399 non-null  str  
 3   home_score  47399 non-null  int64
 4   away_score  47399 non-null  int64
 5   tournament  47399 non-null  str  
 6   city        47399 non-null  str  
 7   country     47399 non-null  str  
 8   neutral     47399 non-null  bool 
dtypes: bool(1), int64(2), str(6)
memory usage: 2.9 MB


In [4]:
df = df_global.copy()

In [5]:
# Quitar espacios o caracteres raros en equipos
df["home_team"] = df["home_team"].str.strip()
df["away_team"] = df["away_team"].str.strip()

In [6]:
# Normalizar Nombres
all_teams = sorted(
    set(df["home_team"]).union(set(df["away_team"]))
)

In [7]:
def normalize_team(name, choices, threshold=90):
    name = str(name).strip()

    match, score, _ = process.extractOne(
        name,
        choices,
        scorer=fuzz.WRatio
    )

    if score >= threshold:
        return match
    else:
        return name

In [8]:
df["home_team"] = df["home_team"].apply(
    lambda x: normalize_team(x, all_teams)
)

In [9]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [26]:
# Peso de partido
def tournament_weigh(tournament):
    t = tournament.lower()
    if "world cup" in t and "qualification" not in t:
        return 1.8
    if "euro" in t or "copa américa" in t or "african cup" in t or "gold cup" in t:
        return 1.4
    if "qualification" in t or "qualifier" in t or "nations league" in t:
        return 1.0
    if "friendly" in t:
        return 0.6
    return 0.3

In [27]:
# Inicializar ratings
initial_elo = 1500
base_k = 20

ratings = {}

In [28]:
# Funciones Elo
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

In [29]:
# Actualización
def update_elo(rating, expected, actual, k=20):
    return rating + k * (actual - expected)

In [30]:
# Guardamos Elo antes del partido
elo_home_list = []
elo_away_list = []
elo_diff_list = []

In [31]:
# Recorrer todos los partidos
for _, row in df.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]

    k = base_k * tournament_weigh(row["tournament"])
    
    home_elo = ratings.get(home, initial_elo)
    away_elo = ratings.get(away, initial_elo)

    elo_home_list.append(home_elo)
    elo_away_list.append(away_elo)
    elo_diff_list.append(home_elo - away_elo)

    expected_home = expected_score(home_elo, away_elo)
    expected_away = expected_score(away_elo, home_elo)

    if home_score > away_score:
        actual_home = 1
        actual_away = 0
    elif home_score < away_score:
        actual_home = 0
        actual_away = 1
    else:
        actual_home = 0.5
        actual_away = 0.5

    ratings[home] = update_elo(
        home_elo,
        expected_home,
        actual_home,
        k
    )

    ratings[away] = update_elo(
        away_elo,
        expected_away,
        actual_away,
        k
    )

In [32]:
df["elo_home"] = elo_home_list
df["elo_away"] = elo_away_list
df["elo_diff"] = elo_diff_list

In [33]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,elo_home,elo_away,elo_diff
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1500.000000,1500.000000,0.000000
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1494.000000,1506.000000,-12.000000
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1499.792850,1500.207150,-0.414301
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1500.199996,1499.800004,0.399991


In [34]:
# Selecciones más fuertes segun Elo
ranking = (
    pd.DataFrame(
        ratings.items(),
        columns=["team", "elo"]
    ).sort_values("elo", ascending=False)
)

ranking.head(10)

,team,elo
7,Argentina,1977.152080
41,Spain,1953.361335
12,France,1925.828288
30,Brazil,1902.248355
1,England,1877.695683
85,Colombia,1877.414959
17,Netherlands,1862.951059
11,Belgium,1850.671240
42,Portugal,1848.164348
18,Germany,1841.303813
